# Aboveground Carbon Across Biomes

Compares aboveground carbon density (Mg C/ha) across four contrasting forest types using two independent satellite-derived sources (ESA CCI Biomass and GEDI L4B), restricted to forest-cover pixels identified via unsupervised clustering of AlphaEarth satellite embeddings (cross-referenced with Hansen tree cover to label the forest cluster).

**Zones:** boreal managed forest (Abitibi, Quebec), intact tropical rainforest (Tapajos, Brazil), native temperate forest (Alerce Costero, Chile), and an even-aged Pinus radiata plantation (Biobio, Chile).

**Pipeline:** define zones -> AlphaEarth clustering + forest-mask identification -> carbon from ESA CCI Biomass (forest pixels only) -> carbon from GEDI L4B (forest pixels only) -> cross-source comparison chart -> AlphaEarth zone signatures -> similarity heatmap.

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from zones import ZONES, get_zone_geometry, get_zone_label
from carbon_sources import get_esa_cci_carbon, get_gedi_carbon, mean_carbon_over_zone
from embedding_utils import (
    get_mean_embedding,
    cosine_similarity_matrix,
    cluster_and_identify_forest,
    get_coarse_forest_mask,
)

PROJECT = "your-gee-project-id"
ee.Initialize(project=PROJECT)

## 1. Study zones

Small bounding boxes (~20-25 km) — illustrative placeholders centered on well-known sites for each forest type. Adjust in `src/zones.py` if you have more precise boundaries.

In [ ]:
for key in ZONES:
    geom = get_zone_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    print(f"{key}: {get_zone_label(key)} — {area_ha:,.0f} ha")

## 2. Forest mask per zone (AlphaEarth clustering + Hansen tree cover)

A raw bounding box mixes forest with roads, water, clearings, and secondary cover, biasing the carbon average. This runs unsupervised k-means on the AlphaEarth embedding for each zone, then labels whichever cluster has the highest mean Hansen tree-cover (2000) as 'forest' and keeps only those pixels for the carbon calculations below.

In [ ]:
CLUSTER_YEAR = 2023
N_CLUSTERS = 4

forest_masks = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    mask, treecover_by_cluster, forest_id = cluster_and_identify_forest(
        CLUSTER_YEAR, geom, n_clusters=N_CLUSTERS
    )
    forest_masks[key] = mask
    print(f"{key}: cluster {forest_id} selected as forest "
          f"(mean tree cover by cluster: {treecover_by_cluster})")

## 3. Aboveground carbon — ESA CCI Biomass (2022), forest-fraction masked

Continuous, gap-free 100 m maps. Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0.

In [ ]:
YEAR = 2022  # most recent year available in ESA CCI Biomass v6.0
MIN_FOREST_FRACTION = 0.7

esa_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    esa_raw = get_esa_cci_carbon(YEAR, geom)
    coarse_mask = get_coarse_forest_mask(forest_masks[key], esa_raw, min_fraction=MIN_FOREST_FRACTION)
    carbon_img = esa_raw.updateMask(coarse_mask)
    mean_carbon = mean_carbon_over_zone(carbon_img, geom, scale=100)
    esa_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (ESA CCI Biomass, forest-fraction >= {MIN_FOREST_FRACTION:.0%})")

## 4. Aboveground carbon — GEDI L4B, forest-fraction masked

Spaceborne lidar, aggregated across the mission period (not a single calendar year) — a cross-check against ESA CCI, not a like-for-like year match. Because GEDI L4B's native grid is 1 km, masking uses the true forest FRACTION within each 1 km cell (via `get_coarse_forest_mask`), not a naive nearest-neighbor resample of the 10 m mask — otherwise mixed cells (forest edges, lakes, coastline) get included or excluded almost arbitrarily.

In [ ]:
gedi_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    gedi_raw = get_gedi_carbon(geom)
    coarse_mask = get_coarse_forest_mask(forest_masks[key], gedi_raw, min_fraction=MIN_FOREST_FRACTION)
    carbon_img = gedi_raw.updateMask(coarse_mask)
    mean_carbon = mean_carbon_over_zone(carbon_img, geom, scale=1000)
    gedi_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (GEDI L4B, forest-fraction >= {MIN_FOREST_FRACTION:.0%})")

## 5. Result 1: cross-source carbon comparison

The headline chart for the LinkedIn post — two independent sources, same four zones, forest-masked.

In [ ]:
comparison_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES],
    "ESA CCI Biomass": [esa_results[k] for k in ZONES],
    "GEDI L4B": [gedi_results[k] for k in ZONES],
})
comparison_df.to_csv("../figures/carbon_comparison.csv", index=False)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width/2, comparison_df["ESA CCI Biomass"], width, label="ESA CCI Biomass", color="forestgreen")
ax.bar(x + width/2, comparison_df["GEDI L4B"], width, label="GEDI L4B", color="saddlebrown")

ax.set_ylabel("Aboveground carbon (Mg C/ha)")
ax.set_title("Aboveground carbon across biomes — forest-masked, two independent sources")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["zone"], rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/carbon_comparison.png", dpi=200)
plt.show()

## 6. AlphaEarth zone signatures

A complementary structural check: how distinct are these four sites in AlphaEarth's 64-dimensional embedding space? An even-aged plantation is expected to look more internally uniform and more different from a structurally complex native forest than two native forests would look from each other.

In [ ]:
EMBEDDING_YEAR = 2023

embeddings = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    embeddings[key] = get_mean_embedding(EMBEDDING_YEAR, geom)
    print(f"{key}: embedding vector computed ({len(embeddings[key])} dims)")

## 7. Result 2: zone similarity heatmap for LinkedIn

In [ ]:
labels, sim_matrix = cosine_similarity_matrix(embeddings)
display_labels = [get_zone_label(k) for k in labels]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(sim_matrix, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(display_labels, rotation=30, ha="right")
ax.set_yticklabels(display_labels)
ax.set_title("AlphaEarth embedding similarity between zones")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                 color="white" if sim_matrix[i, j] > 0.5 else "black")

plt.colorbar(im, label="Cosine similarity")
plt.tight_layout()
plt.savefig("../figures/zone_similarity_heatmap.png", dpi=200)
plt.show()

## 8. Combined result for LinkedIn

Headline numbers for the post caption.

In [ ]:
for key in ZONES:
    print(f"{get_zone_label(key)}:")
    print(f"  ESA CCI Biomass: {esa_results[key]:,.1f} Mg C/ha")
    print(f"  GEDI L4B:        {gedi_results[key]:,.1f} Mg C/ha")